# Managing Files and Storage in Neurodesk

**Author**: Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

Understanding where files live in Neurodesk is crucial for productive, stress-free neuroimaging workflows. Many users get confused about the difference between `/home/jovyan` (container-local storage that resets), `~/neurodesktop-storage` (persistent shared storage), and `/tmp` (temporary, session-only files). This tutorial clarifies the Neurodesk file system layout and teaches best practices for file management.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Understand the Neurodesk file system layout and identify the three storage zones
- Use `~/neurodesktop-storage` as your primary, persistent workspace
- Transfer files into and out of Neurodesk using multiple methods
- Organize datasets in BIDS-compatible folder structures within Neurodesk
- Avoid common pitfalls such as saving to `/tmp` or inside containers
:::

## Citation and Resources

### Educational resources

- [Neurodesk documentation](https://neurodesk.org)
- [Brain Imaging Data Structure (BIDS) Specification](https://bids-standard.github.io/bids-standard/)
- [Docker documentation on volumes](https://docs.docker.com/storage/)

## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have a running Neurodesk instance with access to JupyterLab, a file manager, and a terminal. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment (desktop or JupyterLab)
- [ ] Basic familiarity with terminal commands and file paths
- [ ] At least 10 GB free disk space on your host machine for storage
- [ ] (Optional) A BIDS dataset or raw neuroimaging data to organize

## The Neurodesk File System Layout

Neurodesk runs in a **containerized environment** that isolates your tools from your operating system. This isolation is powerful — it means your Neurodesk tools stay compatible no matter your OS — but it also creates different storage zones with different lifespans.

### Three Storage Zones

| Zone | Path | Persistence | Use Case |
|------|------|-------------|----------|
| **Container-local** | `/home/jovyan` | Resets on container restart | Temporary configs, session-specific work |
| **Persistent Shared** | `~/neurodesktop-storage` | Persists forever (bind-mounted from host) | **Primary workspace — save all your data here** |
| **Temporary** | `/tmp` | Wiped on container restart | Scratch files, intermediate results |

![The three storage zones in Neurodesk](/static/tutorials/about_neurodesk/storage/neurodesk_storage_layout.png)
*The Neurodesk containerized architecture and three storage zones: container-local (orange), persistent shared (green), and temporary (red).*

### Zone 1: `/home/jovyan` — Container-Local Storage

When you start Neurodesk, the container creates a fresh `/home/jovyan` directory. This is where JupyterLab saves its configuration, notebooks, and other files. However, **everything in `/home/jovyan` is deleted when the container restarts**.

**Key points:**
- Suitable for temporary notebooks or scratch code
- Do not save important data here
- JupyterLab config (themes, extensions) is reset on restart

**Example:** If you create a notebook at `/home/jovyan/my_analysis.ipynb` and then restart Neurodesk, the file is gone.

### Zone 2: `~/neurodesktop-storage` — Persistent Shared Storage

This is the **single most important directory in Neurodesk**. `~/neurodesktop-storage` is **bind-mounted** from your host machine, meaning:

1. Files saved here persist forever — they don't disappear on restart
2. Files are visible to both Neurodesk (inside the container) and your host OS (outside the container)
3. You can work with files from your native operating system at the same time

**Key points:**
- **Always save your data here**
- This is your workspace for BIDS datasets, raw data, and analysis results
- Files are automatically backed up if you backup your host machine
- Shared with the host — you can edit files on your computer and Neurodesk sees them immediately

**Example:** Save a BIDS dataset to `~/neurodesktop-storage/my-study/` and it will still be there after you restart Neurodesk and later shut down your computer.

### Zone 3: `/tmp` — Temporary Scratch Space

The `/tmp` directory is a fast, ephemeral workspace for intermediate files during analysis. Everything in `/tmp` is **deleted when the container restarts**.

**Key points:**
- Use for large intermediate files that are expensive to recompute
- Never save final results here
- Fast because it often uses RAM or fast disk
- Useful for HPC workflows with `$TMPDIR`

**Example:** An image processing pipeline might save intermediate images to `/tmp/working/` and then move only the final result to `~/neurodesktop-storage/`.

## Checking Available Storage Space

Before uploading large datasets, check how much space is available in `~/neurodesktop-storage` and how much you are already using.

### Using the Terminal

Open a terminal in Neurodesk and run:

```bash
df -h ~/neurodesktop-storage
```

This shows the filesystem mount point and available disk space. You will see output like:

```
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n1p2  500G  120G  380G  24% /
```

To see how much space your current data occupies:

```bash
du -sh ~/neurodesktop-storage/
```

![Terminal output showing disk usage](/static/tutorials/about_neurodesk/storage/terminal_disk_usage.png)
*Terminal showing disk space availability with `df -h` and current usage with `du -sh`.*

## Uploading Files from Your Computer

You have several options to transfer files from your computer into Neurodesk.

### Method 1: JupyterLab File Browser (Easiest)

The simplest method for small files is to use JupyterLab's built-in file browser.

1. Open JupyterLab
2. Navigate to `neurodesktop-storage` in the left sidebar
3. Click the **Upload Files** button (or drag and drop files onto the folder)
4. Select the file(s) from your computer
5. Wait for the upload to complete

**Pros:** Intuitive, works for small to medium files, no command-line needed.

**Cons:** Slow for large datasets, browser may time out.

![JupyterLab file upload button in the toolbar](/static/tutorials/about_neurodesk/storage/jupyter_upload.png)
*The JupyterLab file browser with the upload button highlighted. Drag files here or click the button.*

### Method 2: Neurodesktop File Manager

The Neurodesktop desktop environment has a native file manager that can access `~/neurodesktop-storage` directly.

1. Click the file manager icon on the Neurodesktop taskbar
2. Navigate to the `neurodesktop-storage` folder
3. Copy/paste files from your host drive

**Pros:** Fast, works with the host operating system directly.

**Cons:** Requires Neurodesktop (not available in JupyterLab-only mode).

### Method 3: Terminal with `scp` (For Remote Instances)

If Neurodesk is running on a remote machine or server, use `scp` from your local terminal:

```bash
# Upload a single file
scp /path/to/local/file.nii.gz user@neurodesk-server:~/neurodesktop-storage/

# Upload an entire directory recursively
scp -r /path/to/local/dataset/ user@neurodesk-server:~/neurodesktop-storage/
```

Replace `neurodesk-server` with your Neurodesk host IP or hostname, and `user` with your login name.

**Pros:** Fast for large files, supports parallel transfers with `-P` flag.

**Cons:** Requires command line, network latency depends on connection speed.

## Downloading Files to Your Computer

Once you have processed data in Neurodesk, you probably want to move the results back to your computer.

### Method 1: JupyterLab Right-Click Download

1. In JupyterLab, right-click the file you want to download
2. Select **Download**
3. Your browser downloads the file to the default Downloads folder

**Pros:** Simple, no command-line needed.

**Cons:** Single files only, slow for large files, browser may time out on files >100 MB.

### Method 2: Terminal with `scp` (Remote Systems)

From your local machine terminal:

```bash
# Download a single file
scp user@neurodesk-server:~/neurodesktop-storage/results.nii.gz /local/path/

# Download a directory
scp -r user@neurodesk-server:~/neurodesktop-storage/my-results/ /local/path/
```

**Pros:** Fast, reliable for large files.

**Cons:** Requires command-line access, network-dependent.

### Method 3: Cloud Storage Integration

For large datasets or remote collaboration, use cloud storage from within Neurodesk:

```bash
# Example: Install rclone and sync to AWS S3
apt-get update && apt-get install -y rclone
rclone config  # Configure your S3 credentials
rclone sync ~/neurodesktop-storage/results/ s3:my-bucket/results/
```

Alternatively, use the [OSF client](https://osf.io/) or [DataLad](https://www.datalad.org/) for dataset sharing.

**Pros:** Reliable, supports large files, enables collaboration.

**Cons:** Requires cloud account setup, incurs storage costs.

## Organizing Your Data: BIDS Structure

The [Brain Imaging Data Structure (BIDS)](https://bids-standard.github.io/bids-standard/) is the community standard for organizing neuroimaging data. Organizing your data in BIDS format makes it:
- Easier to understand and share
- Compatible with automated analysis pipelines
- Suitable for publication and data sharing

### BIDS Folder Structure

A typical BIDS dataset in `~/neurodesktop-storage/` looks like:

```
~/neurodesktop-storage/my-bids-dataset/
├── sub-001/
│   ├── ses-01/
│   │   ├── anat/
│   │   │   ├── sub-001_ses-01_T1w.nii.gz
│   │   │   └── sub-001_ses-01_T1w.json
│   │   ├── func/
│   │   │   ├── sub-001_ses-01_task-rest_bold.nii.gz
│   │   │   └── sub-001_ses-01_task-rest_bold.json
│   │   └── dwi/
│   │       ├── sub-001_ses-01_dwi.nii.gz
│   │       ├── sub-001_ses-01_dwi.bval
│   │       └── sub-001_ses-01_dwi.bvec
├── sub-002/
├── sub-003/
├── README
├── CHANGES
├── dataset_description.json
└── participants.tsv
```

### Creating a BIDS Dataset

You can generate the required metadata files with the [BIDS Starter Kit](https://bids-standard.github.io/bids-starter-kit/tutorials/dataset_setup.html) or tools like [dcm2niix](https://github.com/rordenlab/dcm2niix) (pre-installed in Neurodesk).

**Key benefits:**
- Standardized file naming prevents accidental overwrites
- JSON sidecars store acquisition parameters automatically
- Compatible with tools like [fMRIPrep](https://fmriprep.org/), [QSMxT](https://qsmxt-recon.readthedocs.io/), and [MRIQC](https://mriqc.readthedocs.io/)

## Working with Large Datasets

Neurodesk is designed to handle large neuroimaging datasets. Follow these tips to work efficiently with multi-terabyte datasets.

### Tip 1: Use Symbolic Links Instead of Copies

If your raw data lives in one location and you want to reference it from a processing directory, use a symbolic link (symlink) instead of copying the entire dataset:

```bash
ln -s ~/neurodesktop-storage/raw-data ~/neurodesktop-storage/analysis/input-data
```

This creates a pointer to the original data without duplicating it. You can access files as if they were in the analysis folder, but they take up no extra disk space.

**Pros:** Saves disk space, keeps one copy of data, quick to set up.

**Cons:** If the original data is deleted, the symlink breaks.

### Tip 2: Lazy Data Fetching with DataLad

For very large datasets (TB+), use [DataLad](https://www.datalad.org/) to fetch only the files you need:

```bash
# Install DataLad
pip install datalad

# Clone a dataset (downloads only metadata)
datalad clone https://github.com/OpenNeuroDatasets/ds004748 ~/neurodesktop-storage/ds004748

# Fetch only specific subjects
datalad get ~/neurodesktop-storage/ds004748/sub-01/anat/
```

DataLad is especially useful for remote HPC storage or the [OpenNeuro](https://openneuro.org/) data archive.

**Pros:** Fetch only what you need, reduces bandwidth and storage.

**Cons:** Requires git knowledge, slower initial queries.

### Tip 3: Monitor Disk Usage

Regularly check which folders are taking up the most space:

```bash
# Top-level disk usage
du -sh ~/neurodesktop-storage/*/

# Find largest files
find ~/neurodesktop-storage -type f -size +1G -exec du -h {} \; | sort -rh | head -20

# Real-time disk usage monitoring
ncdu ~/neurodesktop-storage
```

The `ncdu` tool provides an interactive interface to explore disk usage.

### Tip 4: Compress Intermediate Results

Neuroimaging data compresses well. Save disk space by compressing intermediate results:

```bash
# Compress a single file
gzip ~/neurodesktop-storage/results/intermediate.nii
# Result: intermediate.nii.gz

# Compress an entire directory
tar -czf ~/neurodesktop-storage/results/archive.tar.gz ~/neurodesktop-storage/results/working/
```

NIfTI files typically compress to 30–50% of their original size.

## File Permissions and Common Pitfalls

Working across the container boundary can sometimes cause unexpected permission issues. Here is what you need to know.

### Common Pitfall 1: Files Saved to `/home/jovyan` Disappear on Restart

**Problem:** You save a notebook or dataset to `/home/jovyan/` and after restarting Neurodesk, the files are gone.

**Solution:** Always save to `~/neurodesktop-storage/`. You can create symlinks from `/home/jovyan` if you want quick access, but the actual files should live in `neurodesktop-storage`.

```bash
ln -s ~/neurodesktop-storage/my-analysis ~/my-analysis
```

### Common Pitfall 2: Using `/tmp` for Important Results

**Problem:** You save analysis results to `/tmp/` thinking they are safe, but the next time Neurodesk restarts (or after a period of inactivity), all files in `/tmp` are deleted.

**Solution:** Use `/tmp` only for scratch files and temporary computation. Move final results to `~/neurodesktop-storage/`:

```bash
# Process data in /tmp (fast)
fsl_process /tmp/working/ /tmp/output.nii.gz

# Move final result to persistent storage
mv /tmp/output.nii.gz ~/neurodesktop-storage/results/final_output.nii.gz
```

### Common Pitfall 3: File Ownership Issues

**Problem:** You create files inside a container, and when you try to delete or modify them from your host OS, you get "Permission denied".

**Explanation:** Files created inside the Neurodesk container may be owned by the container's user ID (UID), which differs from your host user ID. This mismatch can cause permission conflicts.

**Solution:**
1. Save all files to `~/neurodesktop-storage/`, which is bind-mounted with correct permissions
2. If you encounter permission issues, use `chown` to fix ownership (requires sudo):
   ```bash
   sudo chown -R $USER ~/neurodesktop-storage/
   ```
3. Avoid saving files elsewhere in the container

### Common Pitfall 4: Forgetting to Copy Files Out Before Shutdown

**Problem:** You create results in `/home/jovyan` and then shut down Neurodesk without moving them to `neurodesktop-storage`. The files are lost.

**Solution:** Always use `~/neurodesktop-storage/` as your primary workspace. As a safety net, regularly run:

```bash
# Copy everything from /home/jovyan to persistent storage
cp -r ~/work/* ~/neurodesktop-storage/backup/
```

## Summary

In this tutorial you learned:

1. **The three storage zones** in Neurodesk: container-local (`/home/jovyan`), persistent shared (`~/neurodesktop-storage`), and temporary (`/tmp`)
2. **Where to save your data**: always use `~/neurodesktop-storage/` for all important files
3. **How to transfer files**: multiple methods including JupyterLab upload, terminal `scp`, and cloud storage
4. **How to organize data**: BIDS format enables compatibility with analysis pipelines
5. **Best practices for large datasets**: symlinks, DataLad, disk monitoring, and compression
6. **Common pitfalls and how to avoid them**: don't use `/tmp` for final results, remember that `/home/jovyan` resets on restart

:::{seealso}
- [Accessing Neurodesk Tools](accessing_neurodesk_tools.ipynb) — how to find and load analysis tools
- [Introduction to DataLad](datalad.ipynb) — for managing large datasets efficiently
- [BIDS Specification](https://bids-standard.github.io/bids-standard/) — detailed standard for data organization
:::